# LSTM Model Training

End-to-end notebook to train and save the dynamic gesture LSTM model.

## 1) Install and Import Dependencies

In [ ]:
# %pip install -q tensorflow numpy scikit-learn matplotlib seaborn

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

print('TensorFlow version:', tf.__version__)
print('Time:', datetime.now().isoformat(timespec='seconds'))

## 2) Load Dataset

In [ ]:
project_root = Path.cwd()

x_path = project_root / 'data' / 'X_data.npy'
y_path = project_root / 'data' / 'y_data.npy'
labels_path = project_root / 'models' / 'wlasl_labels.npy'

if not x_path.exists() or not y_path.exists():
    raise FileNotFoundError('Run preprocessing first to generate data/X_data.npy and data/y_data.npy')

X = np.load(x_path)
y = np.load(y_path)
class_names = np.load(labels_path, allow_pickle=True) if labels_path.exists() else None

print('X:', X.shape, 'y:', y.shape)
print('class names available:', class_names is not None)

## 3) Explore and Validate Data

In [ ]:
print('Samples:', X.shape[0])
print('Sequence length:', X.shape[1])
print('Feature dimension:', X.shape[2])

classes, counts = np.unique(y, return_counts=True)
print('Unique classes:', len(classes))
print('Min class count:', int(counts.min()))
print('Max class count:', int(counts.max()))

plt.figure(figsize=(10, 4))
plt.bar(classes[:40], counts[:40])
plt.title('Class Distribution (first 40 classes)')
plt.xlabel('Class index')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4) Preprocess Features and Labels

In [ ]:
num_classes = int(len(np.unique(y)))
y_cat = to_categorical(y, num_classes=num_classes)
print('y categorical shape:', y_cat.shape)
print('num_classes:', num_classes)

## 5) Split Data into Train and Validation Sets

In [ ]:
stratify_target = y if np.bincount(y).min() >= 2 else None
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=stratify_target,
)
print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)

## 6) Define Model Architecture

In [ ]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.3),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax'),
])
model.summary()

## 7) Configure Loss, Optimizer, and Metrics

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
print('Model compiled and callback configured.')

## 8) Train the Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1,
)
print('Epochs ran:', len(history.history['loss']))

## 9) Evaluate Model Performance

In [ ]:
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
y_true = np.argmax(y_val, axis=1)
y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)

print('Validation loss:', float(val_loss))
print('Validation accuracy:', float(val_acc))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, cmap='Blues', cbar=True)
plt.title('Validation Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

target_names = [str(x) for x in class_names] if class_names is not None and len(class_names) == num_classes else [f'class_{i}' for i in range(num_classes)]
report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)
print('Macro avg:', {k: round(v, 4) for k, v in report['macro avg'].items()})

## 10) Save Trained Model and Artifacts

In [ ]:
models_dir = project_root / 'models'
reports_dir = project_root / 'reports'
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / 'gesture_model.h5'
labels_out = models_dir / 'wlasl_labels.npy'
metadata_path = reports_dir / f"lstm_training_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

model.save(model_path)
np.save(labels_out, np.array(target_names))

history_dict = {k: [float(x) for x in v] for k, v in history.history.items()}
metadata = {
    'created_at': datetime.now().isoformat(),
    'x_path': str(x_path),
    'y_path': str(y_path),
    'model_path': str(model_path),
    'labels_path': str(labels_out),
    'input_shape': [int(X.shape[1]), int(X.shape[2])],
    'num_classes': int(num_classes),
    'validation_loss': float(val_loss),
    'validation_accuracy': float(val_acc),
    'epochs_ran': int(len(history_dict.get('loss', []))),
    'history': history_dict,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

summary = {
    'samples': int(X.shape[0]),
    'num_classes': int(num_classes),
    'validation_accuracy': float(val_acc),
    'model_path': str(model_path),
    'metadata_path': str(metadata_path),
}
summary